In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while not (project_root / 'data' / 'raw').is_dir() and project_root != project_root.parent:
    project_root = project_root.parent

raw_data_dir = project_root / 'data' / 'raw'
processed_data_dir = project_root / 'data' / 'processed'
processed_data_dir.mkdir(parents=True, exist_ok=True)

# Load telemetry and shop lookup data
electrical_current_df = pd.read_csv(raw_data_dir / 'electrical_current.csv')
robot_shop_df = pd.read_csv(raw_data_dir / 'robot_shop.csv')

# Confirm shop_id is a primary key in the shop table and every telemetry key resolves
assert robot_shop_df['shop_id'].is_unique
assert electrical_current_df['shop_id'].notna().all()
assert electrical_current_df['shop_id'].isin(robot_shop_df['shop_id']).all()

# Left merge retains each telemetry sample and adds its shop details
merged_df = pd.merge(
    electrical_current_df,
    robot_shop_df,
    on='shop_id',
    how='left',
    validate='many_to_one',
)

output_path = processed_data_dir / 'electrical_current_with_shop.csv'
merged_df.to_csv(output_path, index=False)

print('Merged shape:', merged_df.shape)
print('Saved:', output_path)
print(merged_df.head())